# 05. Parallel Dispatch — concurrent tool execution

An agent often produces a *batch* of tool calls in a single turn — `read_file(a)`,
`read_file(b)`, `grep(pattern, c)`. Running them one-at-a-time wastes wall time when
they don't depend on each other. `arcrun.parallel_dispatch` is the small piece of
machinery that decides *when* a batch can run concurrently and *how many* calls may
be in flight at once — without ever introducing a race condition the audit log can't
explain.

This notebook is the deep dive on `arcrun/parallel_dispatch.py`. The companion
executor walkthrough (`02-tool-executor.ipynb`) introduces the dispatch step at a
high level and points here for the details.

**You will learn:**
- Why read-only batches are safe to parallelize and state-modifying ones aren't
- The `ClassificationRegistry` Protocol and how it stays decoupled from `arcagent`
- `BatchClassifier` rules — including the shared-path-argument heuristic
- `ParallelDispatcher`, `SequentialDispatcher`, and the high-level `dispatch_batch`
- How submission order is preserved in the result list even when completion order isn't
- Tuning the semaphore (default 10; FIPS guidance 4) and what tradeoffs that buys
- Race-free audit ordering: how the chain stays deterministic under concurrency
- A reproducible benchmark showing the parallel speedup

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

All public symbols for parallel dispatch live under `arcrun.parallel_dispatch`. The
module is intentionally *not* re-exported from `arcrun.__init__` — it's an internal
execution mechanism that strategies wire in, not surface API for end users.

In [ ]:
import arcrun
from arcrun.parallel_dispatch import (
    BatchClassifier,
    BatchVerdict,
    ClassificationRegistry,
    ParallelDispatcher,
    SequentialDispatcher,
    dispatch_batch,
)

print(f"arcrun {arcrun.__version__}")
print(
    "public symbols:",
    [
        "BatchClassifier",
        "BatchVerdict",
        "ClassificationRegistry",
        "ParallelDispatcher",
        "SequentialDispatcher",
        "dispatch_batch",
    ],
)

## 2. Why parallel matters

Three observations motivate the whole module:

1. **Batches are common.** A single LLM turn often produces N>1 tool calls.
2. **Most batches are I/O bound.** Reading files, hitting search APIs, stat'ing
   paths — each call spends most of its wall time waiting on the kernel or network.
3. **Read-only calls don't conflict with each other.** Two `read_file` calls on
   different paths cannot interfere. `read_file("a")` followed by `write("a")`
   absolutely can.

So: pay the engineering cost to parallelize the safe case, fail closed on everything
else. The single design decision worth memorizing — *unknown classification ⇒
sequential* — is the reason this module is safe to ship.

Below we'll mock a tiny tool that sleeps for a fixed duration, then run a batch of 4
of them sequentially and concurrently to see the gap. The benchmark in §10 makes the
claim quantitative.

In [ ]:
import asyncio
import time


async def sleepy_read(_args, _ctx=None):
    """Pretend to read from a slow disk."""
    await asyncio.sleep(0.05)
    return "ok"


async def sequential_demo() -> float:
    t0 = time.perf_counter()
    for _ in range(4):
        await sleepy_read({})
    return time.perf_counter() - t0


async def parallel_demo() -> float:
    t0 = time.perf_counter()
    await asyncio.gather(*(sleepy_read({}) for _ in range(4)))
    return time.perf_counter() - t0


seq_t = await sequential_demo()
par_t = await parallel_demo()
print(f"sequential: {seq_t * 1000:6.1f} ms")
print(f"parallel:   {par_t * 1000:6.1f} ms")
print(f"speedup:    {seq_t / par_t:5.2f}x")

That's `asyncio` raw. The real dispatcher adds three things on top: classification
(which calls *may* run concurrently), bounded concurrency (a semaphore so we don't
fan out 1,000 file handles), and submission-order result preservation (downstream
audit code expects deterministic indexes).

## 3. Classification — read_only vs state_modifying

Every tool gets a classification string. Two values matter to the dispatcher:

- `"read_only"` — the tool *observes* state; running it twice produces the same
  result and never mutates anything else.
- `"state_modifying"` — the tool *changes* something. File writes, shell commands,
  HTTP POSTs, anything irreversible.

Classification lookup goes through the `ClassificationRegistry` Protocol — a
single-method interface arcrun depends on without naming any concrete type.

In [ ]:
import inspect
from arcrun.parallel_dispatch import ClassificationRegistry

print(inspect.getsource(ClassificationRegistry))

It's a `runtime_checkable` `Protocol` so we can `isinstance()`-test it. The actual
implementation (in production) lives in `arcagent.core.tool_registry.ToolRegistry`,
which is *outside* arcrun's dependency graph. arcrun never imports arcagent. This
is the concern split CLAUDE.md insists on: arcrun = loop, arcagent = agent.

In [ ]:
from typing import Any


class FakeRegistry:
    """Minimal ClassificationRegistry implementation for the notebook.

    Stores name -> classification. Returns 'state_modifying' for unknown
    tools — the same fail-closed behavior arcagent's real registry provides.
    """

    def __init__(self, classifications: dict[str, str]) -> None:
        self._map = classifications

    def get_classification(self, name: str) -> str:
        return self._map.get(name, "state_modifying")


registry = FakeRegistry(
    {
        "read_file": "read_only",
        "grep": "read_only",
        "ls": "read_only",
        "stat": "read_only",
        "write_file": "state_modifying",
        "shell": "state_modifying",
    }
)

assert isinstance(registry, ClassificationRegistry), "protocol check"
print("classifications loaded:")
for name in ("read_file", "grep", "write_file", "shell", "unknown_tool"):
    print(f"  {name:14s} -> {registry.get_classification(name)}")

Notice the last line: an unknown tool is reported as `state_modifying`. That single
default keeps every "oh I forgot to register this" mistake from turning into a race
condition. **Fail closed, always.**

## 4. `BatchClassifier` and `BatchVerdict`

`BatchClassifier.classify(calls)` returns a frozen `BatchVerdict(can_parallelize,
reason)`. The decision logic is a four-rule funnel:

1. Empty batch → sequential (`reason="empty_batch"`)
2. Any tool not classified `read_only` → sequential
3. Any two calls share a *path-like* argument value → sequential (implicit dep)
4. Otherwise → parallel

We model tool calls with a tiny stand-in so the section reads cleanly.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class ToolCall:
    """Minimal stand-in for arcllm.types.ToolCall — only the fields the
    classifier and dispatcher actually read.
    """

    name: str
    arguments: dict[str, Any] = field(default_factory=dict)
    id: str = "tc"


classifier = BatchClassifier(registry)


def show(label: str, calls: list[ToolCall]) -> None:
    v = classifier.classify(calls)
    flag = "PARALLEL" if v.can_parallelize else "SEQUENTIAL"
    print(f"{label:35s} -> {flag:11s} ({v.reason})")


show("empty batch", [])
show(
    "two reads, distinct paths",
    [
        ToolCall("read_file", {"path": "/etc/hosts"}),
        ToolCall("read_file", {"path": "/etc/passwd"}),
    ],
)
show(
    "read + write same path",
    [
        ToolCall("read_file", {"path": "/var/log/x"}),
        ToolCall("write_file", {"path": "/var/log/x", "data": "hi"}),
    ],
)
show(
    "read + grep shared path",
    [
        ToolCall("read_file", {"path": "/var/log/x"}),
        ToolCall("grep", {"path": "/var/log/x", "pattern": "q"}),
    ],
)
show(
    "two reads + unknown tool",
    [
        ToolCall("read_file", {"path": "/a"}),
        ToolCall("mystery_tool", {}),
    ],
)

`BatchVerdict` is `frozen=True`, so once we have a verdict it cannot be mutated by
downstream code — handy for putting it directly into audit events.

In [ ]:
verdict = classifier.classify(
    [
        ToolCall("read_file", {"path": "/a"}),
        ToolCall("grep", {"path": "/b", "pattern": "x"}),
    ]
)
print(f"can_parallelize: {verdict.can_parallelize}")
print(f"reason:          {verdict.reason}")

try:
    verdict.reason = "tampered"  # type: ignore[misc]
except Exception as exc:
    print(f"frozen check: {type(exc).__name__}: {exc}")

**Why the path-argument heuristic?** Two read-only calls on the same path are *probably*
fine, but if the same path appears in another batch position (e.g. you mistakenly mixed
in a `write_file` later, or the LLM hallucinated a `read` of a file it intends to
modify next turn), the conservative default avoids a class of subtle bugs at the cost
of a minor parallelism loss. False positives — sequential when parallel was safe — are
acceptable. False negatives are not.

## 5. `SequentialDispatcher` vs `ParallelDispatcher`

Each dispatcher has the same shape: an async `dispatch(calls, runner)` method that
returns `list[tuple[call, result_or_exception]]` in **submission order**. The runner
is a coroutine the *caller* supplies — typically a thin wrapper around
`arcrun.executor.execute_tool_call`. Keeping `runner` injectable lets the dispatcher
stay agnostic about the actual execution pipeline (sandbox checks, validation,
audit emission, etc. all live in the runner).

In [ ]:
async def fake_runner(call: ToolCall) -> tuple[ToolCall, Any]:
    """Pretend to run a tool. Returns (call, result) on success.

    Real runners catch ToolError, validate args, emit audit events, and so on.
    """
    await asyncio.sleep(0.01)
    return call, f"result-of-{call.name}"


calls = [ToolCall(f"tool_{i}") for i in range(3)]

seq_results = await SequentialDispatcher().dispatch(calls, fake_runner)
print("sequential:")
for call, result in seq_results:
    print(f"  {call.name} -> {result}")

In [ ]:
par_results = await ParallelDispatcher(max_parallel=10).dispatch(calls, fake_runner)
print("parallel (max=10):")
for call, result in par_results:
    print(f"  {call.name} -> {result}")

assert [r[0].name for r in par_results] == [c.name for c in calls], (
    "ParallelDispatcher must preserve submission order in the result list"
)
print("\nsubmission-order preservation: OK")

Both dispatchers also handle partial failure: if `runner` raises, the exception
object lands in the result tuple instead of aborting the whole batch (R-023). This
matches `asyncio.gather(return_exceptions=True)` semantics for `ParallelDispatcher`
and is matched explicitly by `SequentialDispatcher` so behavior is symmetric.

In [ ]:
async def flaky_runner(call: ToolCall) -> tuple[ToolCall, Any]:
    if call.name == "explode":
        raise RuntimeError("kaboom")
    return call, f"ok:{call.name}"


calls = [ToolCall("a"), ToolCall("explode"), ToolCall("c")]
results = await ParallelDispatcher(max_parallel=10).dispatch(calls, flaky_runner)
for call, outcome in results:
    if isinstance(outcome, Exception):
        print(f"  {call.name:8s} -> RAISED {type(outcome).__name__}: {outcome}")
    else:
        print(f"  {call.name:8s} -> {outcome}")

## 6. `dispatch_batch` — the end-to-end entry point

`dispatch_batch(calls, runner, *, classifier, max_parallel=10)` is the function
strategies (or anyone integrating arcrun) actually call. It wires the classifier and
the right dispatcher together:

```
verdict = classifier.classify(calls)
if verdict.can_parallelize:
    return await ParallelDispatcher(max_parallel=...).dispatch(...)
return await SequentialDispatcher().dispatch(...)
```

We'll watch it pick the right strategy on two batches that are identical *except*
for the classification of one tool.

In [ ]:
async def runner(call: ToolCall) -> tuple[ToolCall, Any]:
    await asyncio.sleep(0.02)
    return call, f"ran:{call.name}"


async def measure(label: str, calls: list[ToolCall]) -> None:
    t0 = time.perf_counter()
    results = await dispatch_batch(
        calls,
        runner,
        classifier=classifier,
        max_parallel=10,
    )
    elapsed = (time.perf_counter() - t0) * 1000
    names = [r[0].name for r in results]
    print(f"{label:30s} elapsed={elapsed:5.1f} ms  result_order={names}")


parallel_calls = [
    ToolCall("read_file", {"path": "/a"}),
    ToolCall("read_file", {"path": "/b"}),
    ToolCall("grep", {"path": "/c", "pattern": "x"}),
]
sequential_calls = [
    ToolCall("read_file", {"path": "/a"}),
    ToolCall("write_file", {"path": "/b", "data": "y"}),
    ToolCall("grep", {"path": "/c", "pattern": "x"}),
]

await measure("all read_only -> parallel", parallel_calls)
await measure("contains state_modifying", sequential_calls)

Wall-time pattern to expect:
- All read_only: ~20 ms (one sleep, three calls overlap)
- One state_modifying: ~60 ms (three sleeps in series)

Submission order in the result list is identical for both — that's the contract.

In [ ]:
async def observe_events() -> None:
    seen: list[tuple[float, str, str]] = []

    async def trace_runner(call: ToolCall) -> tuple[ToolCall, Any]:
        seen.append((time.perf_counter(), call.name, "start"))
        await asyncio.sleep(0.03)
        seen.append((time.perf_counter(), call.name, "end"))
        return call, "ok"

    calls = [
        ToolCall("read_file", {"path": "/x"}),
        ToolCall("read_file", {"path": "/y"}),
        ToolCall("read_file", {"path": "/z"}),
    ]
    await dispatch_batch(calls, trace_runner, classifier=classifier, max_parallel=10)

    t0 = seen[0][0]
    for ts, name, phase in seen:
        print(f"  +{(ts - t0) * 1000:5.1f} ms  {name}.{phase}")


await observe_events()

All three `start` events fire essentially together, then all three `end` events fire
after one ~30 ms sleep. That's the parallelism shape — N tools in roughly the wall
time of one.

## 7. Submission order — the audit guarantee

`asyncio.gather` returns results in the order coroutines were submitted, regardless
of which one finishes first — `ParallelDispatcher.dispatch` relies on exactly that
guarantee and nothing more. There is no separate `_seq`-tagging mechanism: the
result *list position* is the only ordering signal, and it's sufficient because
downstream audit emission iterates that list rather than reacting to raw completion
order.

In [ ]:
async def seq_aware_runner(call: ToolCall) -> tuple[ToolCall, Any]:
    """Sleep for varied durations so completion order ≠ submission order."""
    delay = call.arguments["delay"]
    await asyncio.sleep(delay)
    return call, {"name": call.name}


calls = [
    ToolCall("slow", {"delay": 0.05}),
    ToolCall("medium", {"delay": 0.02}),
    ToolCall("fast", {"delay": 0.001}),
]

dispatcher = ParallelDispatcher(max_parallel=10)
results = await dispatcher.dispatch(calls, seq_aware_runner)

print("submission order (result list):")
for position, (call, result) in enumerate(results):
    name = result["name"]
    print(f"  position={position}  name={name}")

**Production wiring:** strategies that drive `dispatch_batch` don't need to do
anything extra to get this guarantee — it falls out of `asyncio.gather` for free.
There is no opt-in flag to remember; every `ParallelDispatcher`/`SequentialDispatcher`
call preserves submission order in its result list by construction.

## 8. Semaphore tuning — `max_parallel` and FIPS guidance

`ParallelDispatcher` bounds in-flight calls with `asyncio.Semaphore(max_parallel)`.
The constructor default is **10**. Two reasons for the cap:

- **Resource limits.** Even read-only tools consume file descriptors, sockets,
  thread-pool slots. Unbounded fan-out is how you DOS the host.
- **Fairness.** A single agent shouldn't starve every other coroutine on the
  event loop. The cap throttles one batch in favor of overall responsiveness.

**FIPS guidance** (per CHANGELOG 0.4.0): operate with `max_parallel=4` when running
in FIPS mode. The cap isn't enforced inside the module — it's a deployment-level
configuration decision passed in by the caller. Lower cap → less concurrency, but
tighter operator control over the per-agent fan-out an auditor will see.

In [ ]:
async def measure_peak(max_parallel: int, n: int = 12) -> tuple[int, float]:
    """Run n sleepy calls; track peak concurrency and wall time."""
    state = {"now": 0, "peak": 0}

    async def runner(call: ToolCall) -> tuple[ToolCall, Any]:
        state["now"] += 1
        state["peak"] = max(state["peak"], state["now"])
        await asyncio.sleep(0.02)
        state["now"] -= 1
        return call, "ok"

    calls = [ToolCall(f"t{i}") for i in range(n)]
    t0 = time.perf_counter()
    await ParallelDispatcher(max_parallel=max_parallel).dispatch(calls, runner)
    elapsed = time.perf_counter() - t0
    return state["peak"], elapsed


for cap in (1, 2, 4, 10):
    peak, elapsed = await measure_peak(cap, n=12)
    print(f"  max_parallel={cap:2d}  peak_in_flight={peak:2d}  wall_time={elapsed * 1000:6.1f} ms")

What you should see:
- `max_parallel=1` is sequential (peak=1, wall ~ 12 × 20 ms = 240 ms).
- `max_parallel=4` (FIPS guidance) caps peak at 4 (~3 batches of 4 ⇒ ~60 ms).
- `max_parallel=10` (default) lets all 12 run with peak ≤10 (~40 ms).
- Doubling the cap doesn't double throughput — diminishing returns once the cap
  exceeds the natural batch size.

In [ ]:
# Validation: max_parallel < 1 is rejected
try:
    ParallelDispatcher(max_parallel=0)
except ValueError as exc:
    print(f"caught: {exc}")

## 9. Race-free audit ordering

Concurrency creates a real audit problem: if two tool runs finish in unpredictable
order, naïvely emitted events would record a non-deterministic timeline. Replays
would diverge, hash chains would mismatch — auditors would lose their minds.

arcrun's solution is a single guarantee: `dispatch_batch` (and both dispatchers
directly) always return results in the order *calls were submitted*, regardless of
the order they actually completed in. Downstream audit emission iterates this
result list, so the per-call audit envelope (`tool_call_started → tool_call_completed`)
is emitted in submission order even though the underlying runtime overlapped.
There's no separate index annotated onto each call — the list position *is* the
canonical order, and it's the only ordering signal a caller needs.

Below: tools complete in reverse order, but the result list — and therefore the
audit emission downstream — stays in submission order.

In [ ]:
completion_log: list[str] = []


async def reverse_runner(call: ToolCall) -> tuple[ToolCall, Any]:
    delay = call.arguments["delay"]
    await asyncio.sleep(delay)
    completion_log.append(call.name)
    return call, {"name": call.name}


calls = [
    ToolCall("first", {"delay": 0.04}),  # submitted first, finishes LAST
    ToolCall("second", {"delay": 0.02}),
    ToolCall("third", {"delay": 0.005}),  # submitted last, finishes FIRST
]

results = await ParallelDispatcher(max_parallel=10).dispatch(calls, reverse_runner)

print("completion order (non-deterministic timeline):")
print(f"  {completion_log}")
print()
print("result-list order (deterministic — submission order):")
for position, (call, result) in enumerate(results):
    name = result["name"]
    print(f"  position={position}  name={name}")

Even though `third` finished first and `first` finished last, the *result list* is
still `first, second, third`. Downstream code emitting `tool_call_completed` from
this list will produce a deterministic chain — race-free in audit, even though the
underlying runtime was concurrent.

## 10. Benchmark — measurable speedup

Final demo: stack 10 read-only calls, each sleeping 50 ms. Sequentially that's
~500 ms; with default `max_parallel=10` it should drop to ~50 ms (≈10× speedup).
We don't assert exact numbers — wall time is environment-sensitive — but the
*relative ordering* should be obvious:

- `parallel(cap=10)` ≈ `parallel(cap=N)` since the batch fits the cap
- `parallel(cap=4)` (FIPS guidance) ≈ 2.5× the cap=10 time
- `sequential` ≈ N × per-call delay


In [ ]:
N = 10
PER_CALL_MS = 50


async def slow_runner(call: ToolCall) -> tuple[ToolCall, Any]:
    await asyncio.sleep(PER_CALL_MS / 1000)
    return call, "ok"


async def benchmark() -> dict[str, float]:
    out: dict[str, float] = {}

    calls_seq = [ToolCall(f"t{i}") for i in range(N)]
    t0 = time.perf_counter()
    await SequentialDispatcher().dispatch(calls_seq, slow_runner)
    out["sequential"] = (time.perf_counter() - t0) * 1000

    for cap in (4, 10, N):
        calls = [ToolCall(f"t{i}") for i in range(N)]
        t0 = time.perf_counter()
        await ParallelDispatcher(max_parallel=cap).dispatch(calls, slow_runner)
        out[f"parallel(cap={cap})"] = (time.perf_counter() - t0) * 1000

    return out


timings = await benchmark()
baseline = timings["sequential"]
print(f"N={N} calls, each sleeping {PER_CALL_MS} ms\n")
header = "label"
print(f"{header:22s}  wall_ms   speedup")
print("-" * 44)
for label, ms in timings.items():
    speedup = baseline / ms
    print(f"{label:22s}  {ms:7.1f}   {speedup:5.2f}x")

A few things to notice:

- Sequential is roughly `N × PER_CALL_MS` — confirms the per-call delay is real.
- `parallel(cap=10)` ≈ `parallel(cap=N)` — once the cap meets the batch size,
  raising it further buys nothing.
- `parallel(cap=4)` is roughly 2.5× the cap=10 number — three semaphore-batches of
  ~50 ms each, plus a tiny bit of scheduling overhead.
- The speedup doesn't quite hit 10× even at cap=10 — `asyncio.sleep` precision and
  the gather/await machinery cost a few ms each.

If you ever see *no* speedup, you're probably running CPU-bound work in `runner`
without yielding the loop. `parallel_dispatch` is for I/O; CPU-bound work needs a
process pool, not a semaphore.

In [ ]:
# Convenience function for quick what-if exploration
async def what_if(n: int, per_call_ms: int, cap: int) -> float:
    async def r(call: ToolCall) -> tuple[ToolCall, Any]:
        await asyncio.sleep(per_call_ms / 1000)
        return call, "ok"

    calls = [ToolCall(f"t{i}") for i in range(n)]
    t0 = time.perf_counter()
    await ParallelDispatcher(max_parallel=cap).dispatch(calls, r)
    return (time.perf_counter() - t0) * 1000


for cap in (1, 2, 4, 8, 16):
    ms = await what_if(n=16, per_call_ms=30, cap=cap)
    print(f"  N=16, 30 ms each, cap={cap:2d} -> {ms:6.1f} ms")

## 11. Summary

Parallel dispatch is a small module with a sharply scoped responsibility: take a
list of tool calls, decide whether they're safe to run concurrently, and run them
while preserving submission order for downstream audit.

**Key takeaways:**
- Tools are classified `read_only` or `state_modifying`. Unknown ⇒ state_modifying
  (fail-closed).
- `BatchClassifier` produces a `BatchVerdict(can_parallelize, reason)`. The four
  rules: empty batch, any non-read-only, shared-path-arg heuristic, then parallel.
- `ParallelDispatcher` runs read-only batches via `asyncio.gather` bounded by a
  semaphore (`max_parallel` default 10; FIPS deployments use 4).
- `SequentialDispatcher` is the fallback for any non-parallelizable batch — same
  return shape, same partial-failure semantics.
- `dispatch_batch(...)` is the high-level entry: classifier → right dispatcher.
- Result lists are *always* in submission order, regardless of completion order —
  this falls out of `asyncio.gather` directly; there is no separate index or opt-in
  flag to configure.

**Public API exercised:**
`ClassificationRegistry`, `BatchClassifier`, `BatchVerdict`, `ParallelDispatcher`,
`SequentialDispatcher`, `dispatch_batch`.

**Next:** `arcrun/02-tool-executor.ipynb` for the broader execution pipeline
(sandbox checks, jsonschema validation, audit emission) and how `dispatch_batch`
plugs into it. `arcrun/01-core-react.ipynb` for the loop wiring that produces these
tool-call batches in the first place. `arcrun/07-event-chain-verification.ipynb`
for the audit chain verification that consumes submission-ordered emissions.